### 02b - MIA Feature Extraction (Multi-Shadow)

Builds the feature datasets for the attack classifier using **multiple** shadow models.

For each shadow model *i* we score:
- its **own train subset** → `member=1`
- the **shared shadow_test** → `member=0`

All shadow feature rows are concatenated into a single large attack-training set.
Target scoring is unchanged from the single-shadow pipeline.

##### Configurable parameters
- `N_SHADOWS` – must match the value used in `01b`

##### Inputs
- `outputs/models/shadow_encoder_{i}.h5`, `shadow_clf_{i}.h5`
- `outputs/models/shadow_{x1,x2,y}_train_{i}.npy`
- `data/external/clinicalbert/*_shadow_test.npy`
- `outputs/models/{target_encoder,target_clf}.h5`
- `data/external/clinicalbert/*_target_{train,test}.npy`

##### Outputs
- `outputs/results/mia_features_shadow.csv`
- `outputs/results/mia_features_target.csv`

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score


In [ ]:
# ==========================
# CONFIG
# ==========================
N_SHADOWS = 10
MODEL_DIR = 'outputs/models'
EMB_DIR = 'data/external/clinicalbert'
RESULT_DIR = 'outputs/results'
os.makedirs(RESULT_DIR, exist_ok=True)


In [ ]:
# ==========================
# Load target model
# ==========================
target_encoder = load_model(f'{MODEL_DIR}/target_encoder.h5', compile=False)
target_clf     = load_model(f'{MODEL_DIR}/target_clf.h5',     compile=False)
print("Target model loaded.")
print(f"  encoder input shape: {target_encoder.input_shape}")
print(f"  clf input shape:     {target_clf.input_shape}")


In [ ]:
# ==========================
# Load all shadow models
# ==========================
shadow_encoders = []
shadow_clfs = []
for i in range(N_SHADOWS):
    enc = load_model(f'{MODEL_DIR}/shadow_encoder_{i}.h5', compile=False)
    clf = load_model(f'{MODEL_DIR}/shadow_clf_{i}.h5',     compile=False)
    shadow_encoders.append(enc)
    shadow_clfs.append(clf)
    print(f"Loaded shadow model {i}")


In [ ]:
# ==========================
# Load embeddings
# ==========================
# Target side
x1_target_train = np.load(f'{EMB_DIR}/x1_target_train.npy')
x2_target_train = np.load(f'{EMB_DIR}/x2_target_train.npy')
y_target_train  = np.load(f'{EMB_DIR}/y_target_train.npy')
x1_target_test  = np.load(f'{EMB_DIR}/x1_target_test.npy')
x2_target_test  = np.load(f'{EMB_DIR}/x2_target_test.npy')
y_target_test   = np.load(f'{EMB_DIR}/y_target_test.npy')

# Shared shadow test (non-member set for every shadow)
x1_shadow_test = np.load(f'{EMB_DIR}/x1_shadow_test.npy')
x2_shadow_test = np.load(f'{EMB_DIR}/x2_shadow_test.npy')
y_shadow_test  = np.load(f'{EMB_DIR}/y_shadow_test.npy')

print(f"Target train: {x1_target_train.shape[0]} pairs")
print(f"Target test:  {x1_target_test.shape[0]} pairs")
print(f"Shadow test:  {x1_shadow_test.shape[0]} pairs (shared)")


In [ ]:
# ==========================
# Scoring helper
# ==========================
def score_pairs(encoder, clf, x1, x2):
    """Run SNN+MLP pipeline, return sigmoid probabilities."""
    enc1 = encoder.predict(x1, verbose=0)
    enc2 = encoder.predict(x2, verbose=0)
    diff = np.abs(enc1 - enc2)
    return clf.predict(diff, verbose=0).flatten()


In [ ]:
# ==========================
# Build feature tables
# ==========================
def per_pair_loss(probs, y_true, eps=1e-7):
    p = np.clip(probs, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

def build_feature_frame(probs, y_true, member_label, source_name):
    return pd.DataFrame({
        'prob':   probs,
        'loss':   per_pair_loss(probs, y_true),
        'y_true': y_true,
        'member': member_label,
        'source': source_name,
    })


In [ ]:
# ==========================
# Score all shadow models
# ==========================
shadow_frames = []

for i in range(N_SHADOWS):
    print(f"Scoring shadow model {i}...")

    # Load this shadow's private train subset
    x1_tr = np.load(f'{MODEL_DIR}/shadow_x1_train_{i}.npy')
    x2_tr = np.load(f'{MODEL_DIR}/shadow_x2_train_{i}.npy')
    y_tr  = np.load(f'{MODEL_DIR}/shadow_y_train_{i}.npy')

    probs_tr = score_pairs(shadow_encoders[i], shadow_clfs[i], x1_tr, x2_tr)
    probs_te = score_pairs(shadow_encoders[i], shadow_clfs[i], x1_shadow_test, x2_shadow_test)

    shadow_frames.append(
        build_feature_frame(probs_tr, y_tr, member_label=1, source_name=f'shadow_{i}_train')
    )
    shadow_frames.append(
        build_feature_frame(probs_te, y_shadow_test, member_label=0, source_name='shadow_test')
    )

shadow_features = pd.concat(shadow_frames, ignore_index=True)
print(f"\nAggregated shadow features: {len(shadow_features)} rows")
print(f"  members:    {(shadow_features['member']==1).sum()}")
print(f"  non-members:{(shadow_features['member']==0).sum()}")
print("\nMean loss by membership (shadow):")
print(shadow_features.groupby('member')['loss'].agg(['mean', 'std']))


In [ ]:
# ==========================
# Score target model (unchanged from single-shadow pipeline)
# ==========================
probs_target_train = score_pairs(target_encoder, target_clf, x1_target_train, x2_target_train)
probs_target_test  = score_pairs(target_encoder, target_clf, x1_target_test,  x2_target_test)

target_features = pd.concat([
    build_feature_frame(probs_target_train, y_target_train, member_label=1, source_name='target_train'),
    build_feature_frame(probs_target_test,  y_target_test,  member_label=0, source_name='target_test'),
], ignore_index=True)

print(f"Target features: {len(target_features)} rows")
print(f"  members:    {(target_features['member']==1).sum()}")
print(f"  non-members:{(target_features['member']==0).sum()}")
print("\nMean loss by membership (target):")
print(target_features.groupby('member')['loss'].agg(['mean', 'std']))


In [ ]:
# ==========================
# Save outputs (same filenames as single-shadow pipeline)
# ==========================
shadow_features.to_csv(f'{RESULT_DIR}/mia_features_shadow.csv', index=False)
target_features.to_csv(f'{RESULT_DIR}/mia_features_target.csv', index=False)

print("Saved:")
print(f"  {RESULT_DIR}/mia_features_shadow.csv")
print(f"  {RESULT_DIR}/mia_features_target.csv")
